In [1]:
import sys
import os
from pathlib import Path
# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)

import time
import re
import requests
import json
import chromadb
from typing import List, Dict, Any

from configs.setting import settings
from configs.GetConfig import config
from src.a_ingestion.a1_loader import SupabaseDataLoader
from src.b_indexing.b0_vector_db import ChromaVectorDatabase
from src.LLMService import LLMService

In [2]:
SuperbaseDataLoader = SupabaseDataLoader()
products = SuperbaseDataLoader.load_products()
policies = SuperbaseDataLoader.load_policies()

[Loader] Đang tải dữ liệu sản phẩm từ Supabase...
  -> Đã tải batch 0 - 999 (1000 sản phẩm)
  -> Đã tải batch 1000 - 1324 (325 sản phẩm)
[Loader] Đã tải thành công tổng cộng 1325 sản phẩm.
[Loader] Đang tải dữ liệu chính sách từ Supabase...
[Loader] Đã tải thành công 3 chính sách từ database.


In [3]:
# Khởi tạo kết nối ChromaDB
client = ChromaVectorDatabase()

# Tách biệt làm 2 Collection chuyên biệt
product_col = client.get_or_create_collection(name="products_collection")
policy_col = client.get_or_create_collection(name="policies_collection")

print(f"Tổng vector sản phẩm: {product_col.count()}")
print(f"Tổng vector chính sách: {policy_col.count()}")

Tổng vector sản phẩm: 1325
Tổng vector chính sách: 3


# 📚 TÀI LIỆU: FORMAT KHÁC BIỆT GIỮA CÁC LLM PROVIDERS

## 1. STREAMING OUTPUT FORMAT

### GROQ (OpenAI-compatible)
```python
for chunk in response:
    delta = chunk.choices[0].delta
    
    # Text content
    if delta.content:
        print(delta.content)
    
    # Thinking (cho GPT models như gpt-oss-120b)
    if delta.reasoning:
        print(delta.reasoning)
    
    # Tool Calls (streamed piece by piece)
    if delta.tool_calls:
        for tc in delta.tool_calls:
            print(f"Tool: {tc.function.name}")
            print(f"Args: {tc.function.arguments}")
```

### GEMINI (Google GenAI SDK)
```python
for chunk in response:
    if hasattr(chunk, 'candidates') and chunk.candidates:
        for part in chunk.candidates[0].content.parts:
            
            # Text content
            if part.text:
                print(part.text)
            
            # Thinking (flag boolean)
            if part.thought == True:
                print("Thinking:", part.text)
            
            # Tool Calls (format khác OpenAI)
            if part.functionCall:
                print(f"Tool: {part.functionCall.name}")
                print(f"Args: {part.functionCall.args}")
```

### OPENAI (Native)
```python
# Tương tự Groq (OpenAI format)
for chunk in response:
    delta = chunk.choices[0].delta
    
    if delta.content:
        print(delta.content)
    
    # Thinking (cho o1/o3 models)
    if delta.reasoning_content:
        print(delta.reasoning_content)
    
    # Tool Calls
    if delta.tool_calls:
        # ... tương tự Groq
```

### CLAUDE (Anthropic)
```python
# Format khác hoàn toàn - Anthropic Messages API
for chunk in response:
    if chunk.type == "content_block_delta":
        if chunk.delta.type == "text_delta":
            print(chunk.delta.text)
        elif chunk.delta.type == "input_json_delta":
            print(chunk.delta.partial_json)  # Tool call args
    
    # Thinking blocks
    if chunk.type == "content_block_start":
        if chunk.content_block.type == "thinking":
            print("Thinking started...")
```

---

## 2. TOOL CALLING FORMAT

### OpenAI-compatible (Groq, OpenAI)
```python
# Request
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "tinh_cong",
            "description": "Cộng hai số",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number"},
                    "b": {"type": "number"}
                },
                "required": ["a", "b"]
            }
        }
    }
]

# Response (non-streaming)
message.tool_calls = [
    {
        "id": "call_abc123",
        "type": "function",
        "function": {
            "name": "tinh_cong",
            "arguments": '{"a": 150, "b": 350}'
        }
    }
]
```

### Gemini (Google)
```python
# Request (FunctionDeclaration)
tools = [
    {
        "function_declarations": [
            {
                "name": "tinh_cong",
                "description": "Cộng hai số",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "a": {"type": "number"},
                        "b": {"type": "number"}
                    },
                    "required": ["a", "b"]
                }
            }
        ]
    }
]

# Response (functionCall trong part)
part.functionCall = {
    "name": "tinh_cong",
    "args": {"a": 150, "b": 350}  # Note: là object, không phải JSON string
}
```

### Claude (Anthropic)
```python
# Request (tool definition)
tools = [
    {
        "name": "tinh_cong",
        "description": "Cộng hai số",
        "input_schema": {
            "type": "object",
            "properties": {
                "a": {"type": "number"},
                "b": {"type": "number"}
            },
            "required": ["a", "b"]
        }
    }
]

# Response (tool_use block)
content_block = {
    "type": "tool_use",
    "id": "toolu_abc123",
    "name": "tinh_cong",
    "input": {"a": 150, "b": 350}
}
```

---

## 3. THINKING/REASONING FORMAT

### Groq (GPT models)
- **Field**: `delta.reasoning`
- **Format**: String text
- **Models**: `openai/gpt-oss-120b`, `openai/gpt-oss-20b`

### Gemini
- **Field**: `part.thought` (boolean flag)
- **Format**: `part.text` chứa nội dung thinking khi `part.thought == True`
- **Config**: `thinking_config.thinking_level` (MINIMAL, LOW, MEDIUM, HIGH)
- **Models**: `gemma-4-26b-a4b-it`, `gemini-3.5-flash`

### OpenAI (o1/o3 models)
- **Field**: `delta.reasoning_content`
- **Format**: String text
- **Config**: `reasoning_effort` (low, medium, high)
- **Models**: `o1`, `o3`, `o3-mini`

### Claude
- **Field**: `thinking` content block
- **Format**: Separate block trong content
- **Config**: Không có explicit config (built-in)
- **Models**: Claude 3.5 Sonnet, Opus

---

## 4. TỔNG KẾT

| Provider | Streaming Format | Tool Format | Thinking Format |
|----------|-----------------|-------------|-----------------|
| Groq | OpenAI-compatible | OpenAI Function | `delta.reasoning` |
| Gemini | `candidates[0].content.parts` | FunctionDeclaration | `part.thought` flag |
| OpenAI | OpenAI-compatible | OpenAI Function | `delta.reasoning_content` |
| Claude | Anthropic Events | tool_use block | thinking block |

**Lưu ý quan trọng:**
- Groq và OpenAI dùng format giống nhau (OpenAI-compatible)
- Gemini có format riêng biệt hoàn toàn
- Claude cũng có format riêng (Anthropic Messages API)
- LLMService hiện tại chỉ hỗ trợ Groq và Gemini đầy đủ

In [5]:
# =====================================================================
# 🚀 VÍ DỤ AGENT VỚI STREAMING (PHỨC TẠP HƠN NHƯNG THỰC TẾ)
# =====================================================================
# Phiên bản này sử dụng STREAMING để hiển thị real-time output
# 
# 🔍 SỰ KHÁC BIỆT GIỮA CÁC PROVIDER:
# 
# 1. GROQ (OpenAI-compatible):
#    - Streaming: chunk.choices[0].delta.content (text)
#    - Thinking: chunk.choices[0].delta.reasoning (cho GPT models)
#    - Tool Calls: chunk.choices[0].delta.tool_calls (streamed piece by piece)
#    - Format: OpenAI Chat Completions API
# 
# 2. GEMINI:
#    - Streaming: chunk.candidates[0].content.parts (list of parts)
#    - Thinking: part.thought === true (flag boolean)
#    - Tool Calls: part.functionCall (khác format với OpenAI)
#    - Format: Google GenAI SDK (không compatible với OpenAI)
# 
# 3. OPENAI (native):
#    - Tương tự Groq (OpenAI format)
#    - Thinking: reasoning_tokens field (cho o1/o3 models)
# 
# 4. CLAUDE (Anthropic):
#    - Format: Anthropic Messages API
#    - Tool Calls: tool_use blocks trong content
#    - Thinking: thinking blocks (khác với reasoning của OpenAI)

import json
from src.LLMService import LLMService
from configs.GetConfig import config
from configs.setting import settings

# 1. Khởi tạo LLM Service
llm_service = LLMService(settings, config)

# =====================================================================
# 🛠️ STEP 1: ĐỊNH NGHĨA TOOLS (HÀM PYTHON THẬT)
# =====================================================================
def tinh_cong(a: float, b: float) -> float:
    return a + b

def tinh_nhan(a: float, b: float) -> float:
    return a * b

# Map tên hàm → function object
available_tools = {
    "tinh_cong": tinh_cong,
    "tinh_nhan": tinh_nhan
}

# =====================================================================
# 📋 STEP 2: MÔ TẢ TOOLS CHO LLM (JSON SCHEMA)
# =====================================================================
# Đây là chuẩn OpenAI Function Calling - Groq, OpenAI, Claude đều dùng format này
# Gemini có format riêng (FunctionDeclaration) nhưng LLMService sẽ convert
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "tinh_cong",
            "description": "Cộng hai số a và b.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "Số thứ nhất"},
                    "b": {"type": "number", "description": "Số thứ hai"}
                },
                "required": ["a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "tinh_nhan",
            "description": "Nhân hai số a và b.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "Số thứ nhất"},
                    "b": {"type": "number", "description": "Số thứ hai"}
                },
                "required": ["a", "b"]
            }
        }
    }
]

# =====================================================================
# 🤖 STEP 3: AGENT VỚI STREAMING ACCUMULATOR
# =====================================================================
class StreamingMathAgent:
    def __init__(self, llm_service: LLMService):
        self.llm_service = llm_service
        self.model = "openai/gpt-oss-120b"

    def run(self, user_question: str):
        messages = [
            {
                "role": "system",
                "content": "Bạn là trợ lý toán học. Hãy sử dụng các công cụ tính toán được cung cấp để giải quyết bài toán."
            },
            {
                "role": "user",
                "content": user_question
            }
        ]
        
        print(f"👤 Câu hỏi: {user_question}\n")
        
        max_turns = 5
        for turn in range(max_turns):
            print(f"🌀 --- LƯỢT {turn + 1} (STREAMING) ---")
            
            # Gọi LLM với streaming
            response = self.llm_service.call_groq(
                model=self.model,
                messages=messages,
                tools=tools_schema
            )
            
            # ⚠️ ACCUMULATOR PATTERN:
            # Vì streaming gửi từng mảnh (chunk), ta cần tích lũy (accumulate) dữ liệu
            # để ghép lại thành message hoàn chỉnh
            text_content = ""
            tool_calls_dict = {}
            
            # Duyệt qua từng chunk
            for chunk in response:
                delta = chunk.choices[0].delta
                
                # A. Text content → in real-time
                if delta.content:
                    print(delta.content, end="")
                    text_content += delta.content
                
                # B. Tool Calls → tích lũy từng mảnh
                # Groq gửi tool_calls theo từng mảnh (id, name, arguments riêng biệt)
                if delta.tool_calls:
                    for tc_chunk in delta.tool_calls:
                        idx = tc_chunk.index  # index để biết đây là tool_calls thứ mấy
                        
                        if idx not in tool_calls_dict:
                            tool_calls_dict[idx] = {
                                "id": tc_chunk.id,
                                "name": tc_chunk.function.name,
                                "arguments": ""
                            }
                        
                        # Cập nhật ID và tên (thường ở chunk đầu)
                        if tc_chunk.id:
                            tool_calls_dict[idx]["id"] = tc_chunk.id
                        if tc_chunk.function.name:
                            tool_calls_dict[idx]["name"] = tc_chunk.function.name
                            
                        # Cộng dồn arguments (string JSON được gửi từng phần)
                        if tc_chunk.function.arguments:
                            tool_calls_dict[idx]["arguments"] += tc_chunk.function.arguments

            # Format lại tool_calls thành cấu trúc chuẩn OpenAI
            formatted_tool_calls = []
            for idx, tc in tool_calls_dict.items():
                formatted_tool_calls.append({
                    "id": tc["id"],
                    "type": "function",
                    "function": {
                        "name": tc["name"],
                        "arguments": tc["arguments"]
                    }
                })

            # Lưu message của assistant vào history
            agent_msg = {
                "role": "assistant",
                "content": text_content if text_content else None
            }
            if formatted_tool_calls:
                agent_msg["tool_calls"] = formatted_tool_calls
                
            messages.append(agent_msg)
            
            # Execute tools nếu có
            if formatted_tool_calls:
                print("\n\n🔧 LLM yêu cầu gọi Tool...")
                for tc in formatted_tool_calls:
                    func_name = tc["function"]["name"]
                    func_args = json.loads(tc["function"]["arguments"])
                    
                    real_function = available_tools[func_name]
                    
                    print(f"   👉 Chạy: {func_name}({func_args})")
                    result = real_function(**func_args)
                    print(f"   📊 Kết quả: {result}")
                    
                    # Feed kết quả vào history
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc["id"],
                        "name": func_name,
                        "content": str(result)
                    })
                print("🔄 Gửi kết quả lại cho LLM...\n")
                continue
            else:
                print("\n\n✅ --- HOÀN THÀNH ---")
                return text_content

# =====================================================================
# 🚀 STEP 4: CHẠY THỬ
# =====================================================================
streaming_agent = StreamingMathAgent(llm_service)
streaming_agent.run("Hãy tính giúp tôi: (150 + 350) * 10 bằng bao nhiêu?")

👤 Câu hỏi: Hãy tính giúp tôi: (150 + 350) * 10 bằng bao nhiêu?

🌀 --- LƯỢT 1 (STREAMING) ---


🔧 LLM yêu cầu gọi Tool...
   👉 Chạy: tinh_cong({'a': 150, 'b': 350})
   📊 Kết quả: 500
🔄 Gửi kết quả lại cho LLM...

🌀 --- LƯỢT 2 (STREAMING) ---


🔧 LLM yêu cầu gọi Tool...
   👉 Chạy: tinh_nhan({'a': 500, 'b': 10})
   📊 Kết quả: 5000
🔄 Gửi kết quả lại cho LLM...

🌀 --- LƯỢT 3 (STREAMING) ---
(150 + 350) × 10 = 500 × 10 = **5,000**.

✅ --- HOÀN THÀNH ---


'(150\u202f+\u202f350)\u202f×\u202f10\u202f=\u202f500\u202f×\u202f10\u202f=\u202f**5,000**.'

In [3]:
# 1. Import trực tiếp các hàm Tool từ d_tools
from src.d_tools import product_search, policy_search

# ==========================================
# 🔍 TEST 1: Tìm kiếm sản phẩm (Product Search)
# ==========================================
print("=== 💻 CHẠY THỬ PRODUCT SEARCH ===")
product_result = product_search(key_word="laptop HP dưới 20 triệu", limit=3)
print(product_result)

print("\n" + "="*50 + "\n")

# ==========================================
# 📄 TEST 2: Tìm kiếm chính sách (Policy Search)
# ==========================================
print("=== 📋 CHẠY THỬ POLICY SEARCH ===")
policy_result = policy_search(key_word="đổi trả", limit=2)
print(policy_result)


=== 💻 CHẠY THỬ PRODUCT SEARCH ===
Product 1:
- Name: Laptop HP 15-DY2093/2193DX 405F7UA
- Brand: HP
- Price: 14,990,000 VNĐ
- Score: 0.1999
- Details:
Sản phẩm: Laptop HP 15-DY2093/2193DX 405F7UA

Thương hiệu: HP | Danh mục: laptop

Thông tin giá & Kho hàng:
- Giá thực tế: 14,990,000 VNĐ
- Giá gốc niêm yết: 18,990,000 VNĐ
- Mức giảm giá: 21.06%
- Tình trạng: Hết hàng

Thông số kỹ thuật chi tiết:
- Bộ vi xử lý (CPU/Chipset): Intel® Core™ i5-1135G7 (up to 4.2 GHz, 8 MB L3 cache, 4 nhân)
- Dung lượng RAM: 8GB
- Chuẩn/Loại RAM: 8 GB DDR4-2666 MHz RAM (2 x 4 GB)
- Dung lượng lưu trữ: 256 GB PCIe® NVMe™ M.2 SSD
- Kích thước màn hình: 15.6 inches
- Độ phân giải màn hình: 1080 x 1920 pixels (FullHD)
- Công nghệ màn hình: Màn hình chống chói, 250 nits, 45% NTSC
- Loại tấm nền màn hình: Tấm nền IPS
- Dung lượng Pin: 3-cell, 41 Wh Li-ion
- Card đồ họa (GPU/VGA): Intel Iris® Xe Graphics
- Hệ điều hành: WIN 10 - Win
- Trọng lượng: 1.69 kg
- Kích thước thiết bị: 35.85 x 24.2 x 1.79 mm
- Kết nối Blue